In [ ]:
import subprocess
import os
import pandas as pd
import netCDF4
import numpy as np
import glob
import time
import matplotlib.pyplot as plt
import copy
import xarray as xr
from datetime import datetime, timedelta 
import dask
from scipy.interpolate import griddata
#from ocean_c_lab_tools import *
#from celluloid import Camera 
#import PyCO2SYS as csys
import seawater as sw
from roms_regrid import *

In [ ]:
#HAFRO_path='/home/x-uheede/R/HAFRO/Hafro_cruises.xls'
model_grid_path="/home/x-uheede/S/Iceland3_MARBL_2024/P_INPUT/Iceland3_grid.nc"
# Grid parameters, only modify these if grid is made in MATLAB
vert_levels=60
theta_s_model=5
theta_b_model=2
hc_model=300

model_bgc_dia_path1="/anvil/scratch/x-uheede/Iceland3_MARBL_2024_CDR1/Iceland3_MARBL_2024_bgc_dia.202407????????.nc"
model_bgc_dia_path2="/anvil/scratch/x-uheede/Iceland3_MARBL_2024_CDR2/Iceland3_MARBL_2024_bgc_dia.202407????????.nc"
model_bgc_dia_path3="/anvil/scratch/x-uheede/Iceland3_MARBL_2024_CDR3/Iceland3_MARBL_2024_bgc_dia.202407????????.nc"

variables_bgc_dia=['pH_3D','FG_CO2','FG_ALT_CO2','pCO2SURF','pCO2SURF_ALT_CO2']
variables_bgc=['ALK','ALK_ALT_CO2','DIC','DIC_ALT_CO2']
variables_cstar=['hDIC','hDIC_ALT_CO2','FG_CO2','FG_ALT_CO2']

model_bgc_path1="/anvil/scratch/x-uheede/Iceland3_MARBL_2024_CDR1/Iceland3_MARBL_2024_bgc.202407????????.nc"
model_bgc_path2="/anvil/scratch/x-uheede/Iceland3_MARBL_2024_CDR2/Iceland3_MARBL_2024_bgc.202407????????.nc"
model_bgc_path3="/anvil/scratch/x-uheede/Iceland3_MARBL_2024_CDR3/Iceland3_MARBL_2024_bgc.202407????????.nc"

model_cstar_path1="/anvil/scratch/x-uheede/Iceland3_MARBL_2024_CDR1/Iceland3_MARBL_2024_cstar.202407????????.nc"
model_cstar_path2="/anvil/scratch/x-uheede/Iceland3_MARBL_2024_CDR2/Iceland3_MARBL_2024_cstar.202407????????.nc"
model_cstar_path3="/anvil/scratch/x-uheede/Iceland3_MARBL_2024_CDR3/Iceland3_MARBL_2024_cstar.202407????????.nc"

target_depth_levels=[1,2,3,4,5] # Specify depth levels of interest
thinner=4 # specify the temporal frequency of data being read (i.e. no need to read in hourly data)


In [ ]:
from roms_tools import Grid, ROMSOutput

grid = Grid.from_file(
    model_grid_path
)

#Only run this cell if grid is made in MATLAB
grid.update_vertical_coordinate(N=vert_levels, theta_s=theta_s_model, theta_b=theta_b_model, hc=hc_model, verbose=False)


In [ ]:
import xarray as xr
import numpy as np

# -----------------------------------
# Group paths
# -----------------------------------
path_dict = {
    "bgc_dia": [
        model_bgc_dia_path1,
        model_bgc_dia_path2,
        model_bgc_dia_path3,
    ],
    "bgc": [
        model_bgc_path1,
        model_bgc_path2,
        model_bgc_path3,
    ],
    "cstar": [
        model_cstar_path1,
        model_cstar_path2,
        model_cstar_path3,
    ],
}

# -----------------------------------
# Variable lists
# -----------------------------------
var_dict = {
    "bgc_dia": variables_bgc_dia,
    "bgc": variables_bgc,
    "cstar": variables_cstar,  # important difference
}

# -----------------------------------
# Loop and create roms_* and ds_* variables
# -----------------------------------
for key, paths in path_dict.items():
    
    for i, path in enumerate(paths, start=1):
        
        # Create ROMSOutput object
        roms_obj = ROMSOutput(
            grid=grid,
            path=[path],
            use_dask=True,
        )
        
        # Assign roms_* variable (e.g., roms_cstar1)
        globals()[f"roms_{key}{i}"] = roms_obj
        
        # Regrid (handle cstar separately)
        if var_dict[key] is not None:
            ds = roms_obj.regrid(
                depth_levels=target_depth_levels,
                var_names=var_dict[key],
            )
        else:
            ds = roms_obj.regrid(
                depth_levels=target_depth_levels,
            )
        
        # Assign ds_* variable (e.g., ds_cstar1)
        globals()[f"ds_{key}{i}"] = ds

In [ ]:
# =========================
# Store roms_cstar objects
# =========================
roms_cstar = {
    "CDR1": roms_cstar1,
    "CDR2": roms_cstar2,
    "CDR3": roms_cstar3
}

# =========================
# Save NetCDF files
# =========================
for ens, cstar in roms_cstar.items():

    ds_cdr = cstar.cdr_metrics()

    outfile = f"Iceland3_{ens}_cdr_metrics.nc"

    cstar.ds_cdr.to_netcdf(
        outfile
    )

    print(f"Saved {outfile}")